# TypiClust Active Learning on CIFAR-10

Full implementation of the **TPC_RP** active learning pipeline from:

> Hacohen, G., Dekel, O., & Weinshall, D. (2022).  
> **Active Learning on a Budget: Opposite Strategies Suit High and Low Budgets**.  
> *ICML 2022*. https://arxiv.org/abs/2202.02794

**Pipeline overview:**
1. Train SimCLR self-supervised encoder on all 50,000 unlabeled CIFAR-10 images
2. Extract 512-dim L2-normalised backbone features
3. Run TypiClust active learning: cluster features → select most typical unlabeled point per cluster
4. Train ResNet-18 from scratch on the labeled subset; evaluate on test set
5. Repeat for multiple rounds and seeds; compare against random baseline

---
## Section 1: Setup & Imports

In [ ]:
%%time
# ── Install dependencies (uncomment in Google Colab) ──────────────────────
# !pip install -q torch torchvision scikit-learn numpy matplotlib scipy tqdm

# ── Mount Google Drive (uncomment in Colab to persist checkpoints) ────────
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.chdir('/content/drive/MyDrive/machine-learning-coursework-2')

import sys, os
# When running from notebooks/, add the repo root to the path
sys.path.insert(0, os.path.abspath('..'))

# ── Standard library ──────────────────────────────────────────────────────
import random
import json
import time
import warnings
from pathlib import Path
from collections import defaultdict

# ── Scientific stack ──────────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from sklearn.manifold import TSNE

warnings.filterwarnings('ignore')

# ── Project modules ───────────────────────────────────────────────────────
from src.utils     import set_seed, load_cifar10, extract_features, save_results, load_results, log
from src.simclr    import SimCLRModel, train_simclr, load_simclr_model
from src.active_learning import TypiClust, RandomSelection
from src.classifier      import CIFARClassifier

print('Imports OK')

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────
GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)

# ── Device ────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Output directories ────────────────────────────────────────────────────
Path('../results').mkdir(exist_ok=True)
Path('../plots').mkdir(exist_ok=True)
print('Output directories ready.')

---
## Section 2: Load CIFAR-10

In [ ]:
%%time
train_dataset, test_dataset = load_cifar10(root='../data')

CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]
NUM_CLASSES = len(CIFAR10_CLASSES)

print(f'Train : {len(train_dataset):,} images')
print(f'Test  : {len(test_dataset):,} images')
print(f'Classes ({NUM_CLASSES}): {CIFAR10_CLASSES}')

In [ ]:
# ── Visualise one image per class (5 × 2 grid) ───────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('CIFAR-10 — one sample per class', fontsize=14, fontweight='bold')

targets = np.array(train_dataset.targets)
for c, ax in zip(range(NUM_CLASSES), axes.flat):
    idx  = int(np.where(targets == c)[0][0])
    img, _ = train_dataset[idx]
    ax.imshow(img)                      # PIL image, no transform
    ax.set_title(CIFAR10_CLASSES[c], fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.savefig('../plots/cifar10_samples.pdf', bbox_inches='tight')
plt.show()
print('Saved → ../plots/cifar10_samples.pdf')

---
## Section 3: SimCLR Training (Step 1 of TypiClust)

In [ ]:
# ── SimCLR hyper-parameters ───────────────────────────────────────────────
# For a quick Colab run use SIMCLR_EPOCHS = 100; for paper-quality use 500.
SIMCLR_EPOCHS    = 100    # ← set to 500 for publication-quality features
SIMCLR_BATCH     = 512
SIMCLR_LR        = 0.4   # SGD base LR for batch 512 (paper default)
SIMCLR_TEMP      = 0.5
CHECKPOINT_PATH  = '../results/simclr_checkpoint.pt'
FEATURES_TRAIN   = '../results/train_features.npy'
FEATURES_TEST    = '../results/test_features.npy'
LABELS_TRAIN     = '../results/train_labels.npy'
LABELS_TEST      = '../results/test_labels.npy'

In [ ]:
%%time
simclr_model = SimCLRModel()

# ── Train OR load from checkpoint ─────────────────────────────────────────
if Path(CHECKPOINT_PATH).exists():
    log('Found SimCLR checkpoint — loading...')
    simclr_model = load_simclr_model(CHECKPOINT_PATH, device=torch.device('cpu'))
    log('Checkpoint loaded.')
else:
    log(f'No checkpoint found. Training SimCLR for {SIMCLR_EPOCHS} epochs...')
    simclr_model = train_simclr(
        dataset          = train_dataset,
        model            = simclr_model,
        num_epochs       = SIMCLR_EPOCHS,
        batch_size       = SIMCLR_BATCH,
        lr               = SIMCLR_LR,
        temperature      = SIMCLR_TEMP,
        checkpoint_path  = CHECKPOINT_PATH,
        checkpoint_every = 50,
        resume           = True,
        device           = DEVICE,
    )
    log('SimCLR training complete.')

In [ ]:
%%time
# ── Extract 512-dim L2-normalised features ────────────────────────────────
if Path(FEATURES_TRAIN).exists() and Path(FEATURES_TEST).exists():
    log('Loading cached features from disk...')
    train_feats  = np.load(FEATURES_TRAIN)
    train_labels = np.load(LABELS_TRAIN)
    test_feats   = np.load(FEATURES_TEST)
    test_labels  = np.load(LABELS_TEST)
else:
    log('Extracting train features...')
    train_feats, train_labels = extract_features(
        train_dataset, simclr_model, device=DEVICE)
    log('Extracting test features...')
    test_feats, test_labels = extract_features(
        test_dataset, simclr_model, device=DEVICE)

    np.save(FEATURES_TRAIN, train_feats)
    np.save(LABELS_TRAIN,   train_labels)
    np.save(FEATURES_TEST,  test_feats)
    np.save(LABELS_TEST,    test_labels)
    log('Features saved to disk.')

norms = np.linalg.norm(train_feats, axis=1)
print(f'Train features : {train_feats.shape}   norm mean={norms.mean():.4f} (expect ~1.0)')
print(f'Test  features : {test_feats.shape}')

In [ ]:
%%time
# ── t-SNE of SimCLR features coloured by class ────────────────────────────
# Use a stratified subset (500 per class = 5,000 total) for speed.
N_TSNE_PER_CLASS = 500
tsne_idx = []
for c in range(NUM_CLASSES):
    class_idx = np.where(train_labels == c)[0]
    chosen    = np.random.default_rng(GLOBAL_SEED).choice(
        class_idx, size=N_TSNE_PER_CLASS, replace=False)
    tsne_idx.append(chosen)
tsne_idx = np.concatenate(tsne_idx)           # (5000,)

log(f'Running t-SNE on {len(tsne_idx)} samples...')
tsne_emb = TSNE(
    n_components=2, perplexity=40, n_iter=1000,
    random_state=GLOBAL_SEED, n_jobs=-1
).fit_transform(train_feats[tsne_idx])         # (5000, 2)

# Save for reuse in Section 6
np.save('../results/tsne_embeddings.npy', tsne_emb)
np.save('../results/tsne_indices.npy',    tsne_idx)
log('t-SNE complete.')

In [ ]:
CMAP = plt.cm.get_cmap('tab10', NUM_CLASSES)

fig, ax = plt.subplots(figsize=(9, 7))
for c in range(NUM_CLASSES):
    mask = train_labels[tsne_idx] == c
    ax.scatter(tsne_emb[mask, 0], tsne_emb[mask, 1],
               c=[CMAP(c)], s=4, alpha=0.5, rasterized=True, label=CIFAR10_CLASSES[c])

ax.legend(markerscale=3, fontsize=9, loc='upper right',
          ncol=2, framealpha=0.8)
ax.set_title(f't-SNE of SimCLR features (n={len(tsne_idx):,}, {SIMCLR_EPOCHS} epochs)',
             fontsize=13)
ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
ax.axis('off')
plt.tight_layout()
plt.savefig('../plots/tsne_simclr_features.pdf', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → ../plots/tsne_simclr_features.pdf')

---
## Section 4: Active Learning Experiment

**Protocol**
| Setting | Value |
|---|---|
| Budget per round B | 10 |
| AL rounds | 5 |
| Cumulative labels | 10 → 20 → 30 → 40 → 50 |
| Initial labeled set L₀ | empty |
| Classifier | ResNet-18, trained from scratch each round |
| Classifier epochs | 200 |
| Seeds | 3 (set `N_SEEDS=10` for paper-quality variance) |
| Strategies | TypiClust, Random |

In [ ]:
# ── Experiment configuration ──────────────────────────────────────────────
BUDGET_B      = 10     # labels queried per round
N_ROUNDS      = 5      # AL rounds
N_SEEDS       = 3      # set to 10 for lower-variance estimates
CLF_EPOCHS    = 200    # ResNet-18 training epochs per round
CLF_BATCH     = 64
MAX_CLUSTERS  = 500    # TypiClust K_max
SEEDS         = list(range(N_SEEDS))

BUDGETS = [BUDGET_B * (r + 1) for r in range(N_ROUNDS)]   # [10,20,30,40,50]
print(f'Budgets: {BUDGETS}')
print(f'Seeds  : {SEEDS}')
print(f'Total classifier training runs: {len(SEEDS) * N_ROUNDS * 2} (2 strategies)')

In [ ]:
# ── Core AL loop (shared by all strategies) ───────────────────────────────

def run_al_experiment(strategy_name, make_selector_fn, seeds, verbose=True):
    """
    Run active learning experiment across multiple seeds.

    Args:
        strategy_name  : str, used for logging and results keys.
        make_selector_fn: callable(seed) -> selector with .select(B, labeled_idx)
        seeds          : list of int random seeds.

    Returns:
        results: dict keyed by seed, each a list of per-round dicts.
    """
    all_results = {}

    for seed in seeds:
        print(f'\n{'='*60}')
        print(f'  {strategy_name}  |  seed={seed}')
        print(f'{'='*60}')
        set_seed(seed)

        selector       = make_selector_fn(seed)
        clf            = CIFARClassifier(device=DEVICE, seed=seed)
        labeled_idx    = np.array([], dtype=np.int64)   # L0 = empty
        seed_records   = []

        for rnd in range(N_ROUNDS):
            # ── 1. Select B new examples ──────────────────────────────
            new_idx = selector.select(
                BUDGET_B,
                labeled_indices=labeled_idx if len(labeled_idx) > 0 else None
            )
            labeled_idx = np.concatenate([labeled_idx, new_idx]).astype(np.int64)
            budget_now  = len(labeled_idx)

            print(f'\n  Round {rnd+1}/{N_ROUNDS}  |  labeled={budget_now}')
            if verbose:
                selected_classes = [CIFAR10_CLASSES[train_labels[i]] for i in new_idx]
                print(f'  Selected classes: {selected_classes}')

            # ── 2. Train classifier on all labeled examples ───────────
            t0 = time.time()
            train_hist = clf.train(
                labeled_idx, train_dataset,
                epochs=CLF_EPOCHS, batch_size=CLF_BATCH,
                seed=seed, verbose=False
            )
            train_time = time.time() - t0

            # ── 3. Evaluate on full test set ──────────────────────────
            eval_res = clf.evaluate(test_dataset)
            test_acc = eval_res['accuracy']

            print(f'  Test accuracy: {test_acc:.2f}%  '
                  f'(train_time={train_time:.1f}s)')

            # ── 4. Log results ────────────────────────────────────────
            seed_records.append({
                'round'            : rnd + 1,
                'budget'           : int(budget_now),
                'test_accuracy'    : float(test_acc),
                'per_class_acc'    : eval_res['per_class_acc'].tolist(),
                'selected_indices' : new_idx.tolist(),
                'final_train_loss' : float(train_hist['train_loss'][-1]),
                'final_train_acc'  : float(train_hist['train_acc'][-1]),
            })

        all_results[seed] = seed_records

    return all_results

In [ ]:
%%time
# ── Run TypiClust ─────────────────────────────────────────────────────────
print('Running TypiClust...')

def make_typiclust(seed):
    return TypiClust(
        features         = train_feats,
        max_clusters     = MAX_CLUSTERS,
        min_cluster_size = 5,
        seed             = seed,
    )

results_typiclust = run_al_experiment('TypiClust', make_typiclust, SEEDS)
save_results({'results': results_typiclust}, '../results/typiclust_results.json')
print('\nTypiClust done.')

In [ ]:
%%time
# ── Run Random baseline ───────────────────────────────────────────────────
print('Running Random baseline...')

def make_random(seed):
    return RandomSelection(train_feats, seed=seed)

results_random = run_al_experiment('Random', make_random, SEEDS)
save_results({'results': results_random}, '../results/random_results.json')
print('\nRandom baseline done.')

---
## Section 5: Results

In [ ]:
# ── Aggregate results across seeds ────────────────────────────────────────

def aggregate(results_dict):
    """
    Returns arrays of shape (n_rounds,) for mean and std across seeds.
    Also returns the full (n_seeds, n_rounds) matrix for statistical tests.
    """
    seeds_list = sorted(results_dict.keys())
    # shape (n_seeds, n_rounds)
    acc_matrix = np.array([
        [r['test_accuracy'] for r in results_dict[s]]
        for s in seeds_list
    ])
    budgets = [r['budget'] for r in results_dict[seeds_list[0]]]
    return {
        'mean'    : acc_matrix.mean(axis=0),
        'std'     : acc_matrix.std(axis=0, ddof=1) if len(seeds_list) > 1 else np.zeros(N_ROUNDS),
        'matrix'  : acc_matrix,
        'budgets' : budgets,
    }

agg_tc  = aggregate(results_typiclust)
agg_rnd = aggregate(results_random)

In [ ]:
# ── Print results table ───────────────────────────────────────────────────
header = f"{'Budget':>8}  {'TypiClust mean':>16}  {'TypiClust std':>14}  "\
         f"{'Random mean':>12}  {'Random std':>11}  {'Δ (TC−Rnd)':>11}"
print(header)
print('─' * len(header))

for i, b in enumerate(agg_tc['budgets']):
    tc_m  = agg_tc['mean'][i]
    tc_s  = agg_tc['std'][i]
    rnd_m = agg_rnd['mean'][i]
    rnd_s = agg_rnd['std'][i]
    delta = tc_m - rnd_m
    print(f"{b:>8d}  {tc_m:>14.2f}%  {tc_s:>12.2f}%  "
          f"{rnd_m:>10.2f}%  {rnd_s:>9.2f}%  {delta:>+10.2f}%")

In [ ]:
# ── Per-seed breakdown ────────────────────────────────────────────────────
for seed in SEEDS:
    tc_accs  = [r['test_accuracy'] for r in results_typiclust[seed]]
    rnd_accs = [r['test_accuracy'] for r in results_random[seed]]
    print(f'Seed {seed}  TypiClust: {[f"{a:.1f}" for a in tc_accs]}  '
          f'Random: {[f"{a:.1f}" for a in rnd_accs]}')

---
## Section 6: Plots

In [ ]:
# ── Publication style ─────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family'    : 'serif',
    'font.size'      : 11,
    'axes.titlesize' : 12,
    'axes.labelsize' : 11,
    'legend.fontsize': 10,
    'figure.dpi'     : 120,
})
TC_COLOR  = '#1f77b4'   # blue
RND_COLOR = '#ff7f0e'   # orange

In [ ]:
# ── Plot 1: Accuracy vs. cumulative budget (line plot with std shading) ───
fig, ax = plt.subplots(figsize=(7, 4.5))

budgets = np.array(agg_tc['budgets'])

for agg, label, color in [
    (agg_tc,  'TypiClust', TC_COLOR),
    (agg_rnd, 'Random',    RND_COLOR),
]:
    mu  = agg['mean']
    std = agg['std']
    ax.plot(budgets, mu, marker='o', linewidth=2, color=color, label=label)
    ax.fill_between(budgets, mu - std, mu + std,
                    alpha=0.18, color=color)

ax.set_xlabel('Cumulative labeled budget')
ax.set_ylabel('Test accuracy (%)')
ax.set_title('TypiClust vs. Random — CIFAR-10')
ax.set_xticks(budgets)
ax.legend(framealpha=0.9)
ax.grid(True, linestyle='--', alpha=0.4)
ax.margins(x=0.05)

plt.tight_layout()
plt.savefig('../plots/accuracy_vs_budget.pdf', bbox_inches='tight')
plt.show()
print('Saved → ../plots/accuracy_vs_budget.pdf')

In [ ]:
# ── Plot 2: Bar chart at final budget (B=50) with error bars ──────────────
final_idx = -1   # last round
methods   = ['TypiClust', 'Random']
means     = [agg_tc['mean'][final_idx], agg_rnd['mean'][final_idx]]
stds      = [agg_tc['std'][final_idx],  agg_rnd['std'][final_idx]]
colors    = [TC_COLOR, RND_COLOR]

fig, ax = plt.subplots(figsize=(4.5, 4.5))
bars = ax.bar(methods, means, color=colors, width=0.45,
              yerr=stds, capsize=6, error_kw=dict(linewidth=1.5))

# Value labels on bars
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + s + 0.4,
            f'{m:.2f}%', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Test accuracy (%)')
ax.set_title(f'Final accuracy at budget={budgets[final_idx]}')
ax.set_ylim(0, max(means) + max(stds) + 8)
ax.grid(True, axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('../plots/final_accuracy_bar.pdf', bbox_inches='tight')
plt.show()
print('Saved → ../plots/final_accuracy_bar.pdf')

In [ ]:
# ── Plot 3: t-SNE with selected examples overlaid ─────────────────────────
# Use seed=0 results; mark which training indices were selected each round.
tsne_emb = np.load('../results/tsne_embeddings.npy')
tsne_idx = np.load('../results/tsne_indices.npy')
tsne_set = set(tsne_idx.tolist())

# Collect all selected indices across rounds for seed=0, TypiClust
tc_selected_all = []
for rnd_rec in results_typiclust[0]:
    tc_selected_all.extend(rnd_rec['selected_indices'])
tc_selected_all = np.array(tc_selected_all)

# Find which selected indices are also in the t-SNE subset
tc_in_tsne      = [i for i in tc_selected_all if i in tsne_set]
tsne_idx_list   = tsne_idx.tolist()
tc_local_idx    = [tsne_idx_list.index(i) for i in tc_in_tsne]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, title_suffix in zip(axes, ['', ' — selected (★) highlighted']):
    for c in range(NUM_CLASSES):
        mask = train_labels[tsne_idx] == c
        ax.scatter(tsne_emb[mask, 0], tsne_emb[mask, 1],
                   c=[CMAP(c)], s=4, alpha=0.4, rasterized=True)
    ax.axis('off')

# Right panel: overlay selected points
ax = axes[1]
if tc_local_idx:
    ax.scatter(tsne_emb[tc_local_idx, 0], tsne_emb[tc_local_idx, 1],
               c='red', s=120, marker='*', zorder=5,
               label=f'TypiClust selected ({len(tc_local_idx)} visible)')
    ax.legend(loc='upper right', fontsize=9, framealpha=0.9)

axes[0].set_title('SimCLR feature space (t-SNE)', fontsize=12)
axes[1].set_title('TypiClust selections overlaid (seed=0)', fontsize=12)

# Shared class legend
patches = [mpatches.Patch(color=CMAP(c), label=CIFAR10_CLASSES[c])
           for c in range(NUM_CLASSES)]
fig.legend(handles=patches, loc='lower center', ncol=5,
           fontsize=9, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('t-SNE of SimCLR Features with TypiClust Selections', fontsize=13)
plt.tight_layout()
plt.savefig('../plots/tsne_selected.pdf', bbox_inches='tight')
plt.show()
print('Saved → ../plots/tsne_selected.pdf')

---
## Section 7: Statistical Analysis

In [ ]:
# ── Helper: Cohen's d ─────────────────────────────────────────────────────
def cohens_d(a, b):
    """Compute Cohen's d for two paired samples."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    diff = a - b
    if diff.std(ddof=1) == 0:
        return 0.0
    return float(diff.mean() / diff.std(ddof=1))


def interpret_d(d):
    ad = abs(d)
    if ad < 0.2:  return 'negligible'
    if ad < 0.5:  return 'small'
    if ad < 0.8:  return 'medium'
    return 'large'

In [ ]:
# ── Paired t-tests and Cohen's d at each budget level ─────────────────────
print('Statistical analysis: TypiClust vs. Random  (paired t-test, α=0.05)\n')

if N_SEEDS < 2:
    print('⚠  Only 1 seed — statistical tests require N_SEEDS ≥ 2.  '
          'Re-run with N_SEEDS=3 or more.')
else:
    header = (f"{'Budget':>8}  {'TC mean':>9}  {'Rnd mean':>10}  "
              f"{'t-stat':>8}  {'p-value':>9}  "
              f"{'Signif':>7}  {'Cohen d':>9}  {'Effect':>10}")
    print(header)
    print('─' * len(header))

    stat_rows = []
    for i, b in enumerate(agg_tc['budgets']):
        tc_accs  = agg_tc['matrix'][:, i]    # (n_seeds,)
        rnd_accs = agg_rnd['matrix'][:, i]

        t_stat, p_val = stats.ttest_rel(tc_accs, rnd_accs)
        d             = cohens_d(tc_accs, rnd_accs)
        sig           = '  *' if p_val < 0.05 else ('  ~' if p_val < 0.10 else '')

        print(f"{b:>8d}  {tc_accs.mean():>8.2f}%  {rnd_accs.mean():>9.2f}%  "
              f"{t_stat:>8.3f}  {p_val:>9.4f}  "
              f"{sig:>7s}  {d:>9.3f}  {interpret_d(d):>10s}")

        stat_rows.append({'budget': b, 't_stat': t_stat, 'p_value': p_val,
                          'cohens_d': d, 'significant_005': bool(p_val < 0.05)})

    print('\n  * p < 0.05   ~ p < 0.10')

    save_results({'statistics': stat_rows}, '../results/statistical_analysis.json')
    print('\nStatistical results saved → ../results/statistical_analysis.json')

In [ ]:
# ── p-value plot ──────────────────────────────────────────────────────────
if N_SEEDS >= 2:
    p_vals = [r['p_value'] for r in stat_rows]
    d_vals = [r['cohens_d'] for r in stat_rows]

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

    # p-values
    ax = axes[0]
    ax.plot(budgets, p_vals, marker='s', color=TC_COLOR, linewidth=2)
    ax.axhline(0.05, color='red', linestyle='--', linewidth=1, label='α=0.05')
    ax.axhline(0.10, color='orange', linestyle=':', linewidth=1, label='α=0.10')
    ax.set_xlabel('Budget'); ax.set_ylabel('p-value')
    ax.set_title('Paired t-test p-values')
    ax.set_xticks(budgets); ax.legend()
    ax.grid(True, linestyle='--', alpha=0.4)

    # Cohen's d
    ax = axes[1]
    ax.bar(budgets, d_vals, color=TC_COLOR, width=3.5, alpha=0.8)
    for threshold, label, ls in [(0.2,'small','--'), (0.5,'medium','-'), (0.8,'large',':')]:
        ax.axhline(threshold, color='gray', linestyle=ls, linewidth=1, label=label)
    ax.set_xlabel('Budget'); ax.set_ylabel("Cohen's d")
    ax.set_title("Effect size (Cohen's d)")
    ax.set_xticks(budgets); ax.legend(fontsize=8)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.savefig('../plots/statistical_analysis.pdf', bbox_inches='tight')
    plt.show()
    print('Saved → ../plots/statistical_analysis.pdf')

---
## Section 8: Summary of Results

In [ ]:
# ── Formatted summary table (paper-style) ─────────────────────────────────
sep  = '─' * 66
dsep = '═' * 66

print(dsep)
print(f"{'ACTIVE LEARNING ON CIFAR-10 — TypiClust vs. Random':^66}")
print(f"{'(Hacohen et al., ICML 2022  —  this implementation)':^66}")
print(dsep)
print(f"{'Setup':}")  
print(f"  Encoder     : ResNet-18 (SimCLR, {SIMCLR_EPOCHS} epochs)")
print(f"  Classifier  : ResNet-18 (from scratch, {CLF_EPOCHS} epochs/round)")
print(f"  Budget / rnd: {BUDGET_B}   Rounds: {N_ROUNDS}   Seeds: {N_SEEDS}")
print(sep)
print(f"{'Budget':>8}  │  "
      f"{'TypiClust':^22}  │  "
      f"{'Random':^22}  │  "
      f"{'Δ':^6}")
print(f"{'':>8}  │  "
      f"{'mean ± std (%)':^22}  │  "
      f"{'mean ± std (%)':^22}  │  ")
print(sep)

for i, b in enumerate(agg_tc['budgets']):
    tc_m   = agg_tc['mean'][i];   tc_s  = agg_tc['std'][i]
    rnd_m  = agg_rnd['mean'][i];  rnd_s = agg_rnd['std'][i]
    delta  = tc_m - rnd_m
    tc_str  = f'{tc_m:.2f} ± {tc_s:.2f}'
    rnd_str = f'{rnd_m:.2f} ± {rnd_s:.2f}'
    print(f"{b:>8d}  │  {tc_str:^22}  │  {rnd_str:^22}  │  {delta:>+5.2f}%")

print(dsep)

best_tc   = agg_tc['mean'][-1]
best_rnd  = agg_rnd['mean'][-1]
gain      = best_tc - best_rnd
print(f"  Final budget ({BUDGETS[-1]} labels):")
print(f"    TypiClust : {best_tc:.2f}% ± {agg_tc['std'][-1]:.2f}%")
print(f"    Random    : {best_rnd:.2f}% ± {agg_rnd['std'][-1]:.2f}%")
print(f"    Gain      : {gain:+.2f}%")
print(dsep)

In [ ]:
# ── Per-class accuracy at final budget (seed=0) ───────────────────────────
tc_per_class  = np.array(results_typiclust[0][-1]['per_class_acc'])
rnd_per_class = np.array(results_random[0][-1]['per_class_acc'])

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(NUM_CLASSES)
w = 0.38
ax.bar(x - w/2, tc_per_class,  width=w, color=TC_COLOR,  label='TypiClust', alpha=0.9)
ax.bar(x + w/2, rnd_per_class, width=w, color=RND_COLOR, label='Random',    alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(CIFAR10_CLASSES, rotation=30, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title(f'Per-class accuracy at budget={BUDGETS[-1]} (seed=0)')
ax.legend(); ax.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('../plots/per_class_accuracy.pdf', bbox_inches='tight')
plt.show()
print('Saved → ../plots/per_class_accuracy.pdf')

In [ ]:
# ── List all saved outputs ─────────────────────────────────────────────────
import glob
print('Results files:')
for f in sorted(glob.glob('../results/*.json') + glob.glob('../results/*.npy') + glob.glob('../results/*.pt')):
    size = Path(f).stat().st_size
    print(f'  {f}  ({size/1024:.1f} KB)')
print('\nPlot files:')
for f in sorted(glob.glob('../plots/*.pdf') + glob.glob('../plots/*.png')):
    size = Path(f).stat().st_size
    print(f'  {f}  ({size/1024:.1f} KB)')